In [ ]:
import sys
sys.path.append(r"C:\\Users\\Lenovo\\Desktop")

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import os
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torchvision.models as models
import matplotlib.pyplot as plt
import numpy as np
import torch.optim as optim
# from torchsummary import summary

In [ ]:
unified_emotion_mapping = {
    "Neutral": 0,
    "Anger": 1,
    "Disgust": 2,
    "Fear": 3,
    "Happy": 4,
    "Sad": 5,
    "Surprise": 6,
}

In [ ]:
emotion_mappingA = {
    0: "Neutral",
    1: "Anger",
    2: "Disgust",
    3: "Fear",
    4: "Happy",
    5: "Sad",
    6: "Surprise",
}
class VideoFrameDatasetA(Dataset):
    def __init__(self, batches_dir_list, annotations_dir, transform=None):
        self.transform = transform
        self.frame_paths = []
        self.labels = []

        # Iterate over each batch directory
        for batch_dir in batches_dir_list:
            video_folders = [d for d in os.listdir(batch_dir) if os.path.isdir(os.path.join(batch_dir, d)) and not d.startswith('.')]

            for video_folder in video_folders:
                video_path = os.path.join(batch_dir, video_folder)
                annotation_file = os.path.join(annotations_dir, f"{video_folder}.txt")

                if not os.path.isfile(annotation_file):
                    continue  # Skip if the annotation file does not exist

                # Load labels from annotation file
                with open(annotation_file, 'r') as f:
                    next(f)  # Skip the first line with emotion names
                    labels_for_video = [int(line.strip()) for line in f]  # Labels in file start from 1

                # Get all frame files, assuming they're named in a sortable manner that reflects their order
                frame_files = sorted([f for f in os.listdir(video_path) if f.endswith(('.jpg', '.png'))])

                if len(labels_for_video) > len(frame_files):
                    labels_for_video = labels_for_video[:len(frame_files)]

                for frame_file, label in zip(frame_files, labels_for_video):
                    if label not in [-1, 7]:
                        frame_path = os.path.join(video_path, frame_file)
                        self.frame_paths.append(frame_path)
                        self.labels.append(label)

    def __len__(self):
        return len(self.frame_paths)

    def __getitem__(self, idx):
        img_path = self.frame_paths[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        original_label = self.labels[idx]  # This is the original integer label
        label_name = emotion_mappingA[original_label]  # Get the label name using the original mapping
        unified_label = unified_emotion_mapping[label_name]  # Convert to unified label
        file_name = os.path.basename(img_path)
        
        # return image, unified_label, file_name
        return image, unified_label


In [ ]:
import pandas as pd
emotion_mappingD = {
    1: "Happy",
    2: "Sad",
    3: "Neutral",
    4: "Anger",
    5: "Surprise",
    6: "Disgust",
    7: "Fear",
}
class VideoFrameDatasetD(Dataset):
    def __init__(self, batches_dir_list, annotations_file, transform=None):
        self.transform = transform
        self.frame_paths = []
        self.labels = []

        # Load annotations into a dictionary
        annotations_df = pd.read_excel(annotations_file)
        annotations = dict(zip(annotations_df['order'], annotations_df['label']))

        # Iterate over all provided batch directories
        for batch_dir in batches_dir_list:
            # Iterate over batch folders
            for video_folder in sorted(os.listdir(batch_dir)):
                if video_folder.startswith('.'):  # Skip hidden/system files like .DS_Store
                    continue
                video_id = int(video_folder)  # Extract video ID from folder name
                video_path = os.path.join(batch_dir, video_folder)
                if not os.path.isdir(video_path) or video_id not in annotations:
                    continue  # Skip any files, and process only directories with a corresponding label

                if annotations[video_id] != 0:
                    for frame_file in sorted(os.listdir(video_path)):
                        frame_path_abs = os.path.join(video_path, frame_file)  # Absolute path for loading the image
                        self.frame_paths.append(frame_path_abs)
                        self.labels.append(annotations[video_id])

    def __len__(self):
        return len(self.frame_paths)

    def __getitem__(self, idx):
        img_path = self.frame_paths[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        original_label = self.labels[idx]  # This is the original integer label
        label_name = emotion_mappingD[original_label]  # Get the label name using the original mapping
        unified_label = unified_emotion_mapping[label_name]  # Convert to unified label
        
        file_name = os.path.basename(img_path)
        
        # return image, unified_label, file_name
        return image, unified_label

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# images_path = '/Users/sisjkly/Downloads/Samples/test'
# annotations_path = '/Users/sisjkly/Downloads/Samples/test.txt'
batches_dir_list_1 = [
    
    "F:\\out\\Takeout\\MyDrive\\original\\output\\batch1",
    "F:\\out\\Takeout\\MyDrive\\original\\output\\batch2",
    "F:\\out\\Takeout\\MyDrive\\original\\output\\batch3-1",
]
batches_dir_list_1_v = [
    "F:\\Aff-Wild2\\Openpose\\valid"
]

annotations_dir_1 = 'F:\Aff-Wild2/5th_ABAW_Annotations/EXPR_Classification_Challenge/Train_Set'
train_dataset_1 = VideoFrameDatasetA(batches_dir_list=batches_dir_list_1,
                                  annotations_dir=annotations_dir_1,
                                  transform=transform)
# train_loader_1 = DataLoader(train_dataset_1, batch_size=32, shuffle=True)
annotations_dir_1_v = 'F:\Aff-Wild2/5th_ABAW_Annotations/EXPR_Classification_Challenge/Validation_Set'
valid_dataset_1 = VideoFrameDatasetA(batches_dir_list=batches_dir_list_1_v,
                                  annotations_dir=annotations_dir_1_v,
                                  transform=transform)
valid_loader = DataLoader(valid_dataset_1, batch_size=64, shuffle=True)






batches_dir_list_2 = [
    
    "F:\\out\\Takeout\\MyDrive\\original\\output\\part1",
    "F:\\out\\Takeout\\MyDrive\\original\\output\\part2",
    "F:\\out\\Takeout\\MyDrive\\original\\output\\part3",
    "F:\\out\\Takeout\\MyDrive\\original\\output\\part4",
    "F:\\out\\Takeout\\MyDrive\\original\\output\\part5",
    "F:\\out\\Takeout\\MyDrive\\original\\output\\part6",
    "F:\\out\\Takeout\\MyDrive\\original\\output\\part7",
    "F:\\out\\Takeout\\MyDrive\\original\\output\\part8",
]
batches_dir_list_2_v = [
    # "F:\\DFEW\\Openpose\\part9",
    # "F:\\DFEW\\Openpose\\part10",
    
    "F:\\out\\Takeout\\MyDrive\\original\\output\\part9",
    "F:\\out\\Takeout\\MyDrive\\original\\output\\part10",
    "F:\\out\\Takeout\\MyDrive\\original\\output\\part11",
]

annotations_dir_2 = 'F:/DFEW/Annotation/annotation.xlsx'
train_dataset_2 = VideoFrameDatasetD(batches_dir_list=batches_dir_list_2,
                                  annotations_file=annotations_dir_2,
                                  transform=transform)
valid_dataset_2 = VideoFrameDatasetD(batches_dir_list=batches_dir_list_2_v,
                                  annotations_file=annotations_dir_2,
                                  transform=transform)
# valid_loader = DataLoader(valid_dataset_2, batch_size=32, shuffle=False)



In [ ]:
from torch.utils.data import ConcatDataset

# Combine datasets
combined_dataset_1 = ConcatDataset([train_dataset_1, train_dataset_2])
# dataloader = DataLoader(combined_dataset_1, batch_size=32, shuffle=True)
dataloader = DataLoader(train_dataset_1, batch_size=64, shuffle=True)
# dataloader = DataLoader(train_dataset_2, batch_size=128, shuffle=True)

# combined_dataset_2 = ConcatDataset([valid_dataset_1, valid_dataset_2])
# validloader = DataLoader(combined_dataset_2, batch_size=128, shuffle=True)

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F

def show_images(images, labels, n_images=5, class_names=None):
    plt.figure(figsize=(15, 10))
    for i in range(min(n_images, len(images))):  # Ensure we do not exceed batch size
        ax = plt.subplot(1, n_images, i + 1)
        # Convert image tensor to PIL for correct color interpretation
        img = F.to_pil_image(images[i])
        plt.imshow(img)
        # Set title to the corresponding label name
        label_idx = labels[i].item()
        label_str = class_names[label_idx] if class_names and label_idx in class_names else str(label_idx)
        plt.title(label_str)
        plt.axis("off")
    plt.show()

# Assuming 'dataloader' is your DataLoader instance and emotion_mappingA/D correctly maps integer labels to class names.
for images, labels in dataloader:
    # Replace 'emotion_mappingA' with the appropriate mapping you intend to use
    show_images(images, labels, n_images=5, class_names=emotion_mappingA)
    break  # Only show the first batch



# def show_images(images, labels, file_names, n_images=5, class_names=None):
#     plt.figure(figsize=(15, 10))
#     for i in range(min(n_images, len(images))):  # Ensure we do not exceed batch size or number of images
#         ax = plt.subplot(1, n_images, i + 1)
#         # Convert image tensor to PIL for correct color interpretation
#         img = F.to_pil_image(images[i])
#         plt.imshow(img)
#         # Set title to the corresponding label name and file name
#         label_idx = labels[i].item() if labels is not None else -1
#         label_str = class_names[label_idx] if class_names and label_idx in class_names else str(label_idx)
#         file_name_str = file_names[i] if file_names and i < len(file_names) else "Unknown"
#         plt.title(f"{label_str}")
#         plt.axis("off")
#         print(file_name_str)
#     plt.show()

# # Modify the loop to unpack file_names as well
# for images, labels, file_names in dataloader:
#     # Replace 'emotion_mappingD' with the appropriate mapping you intend to use
#     show_images(images, labels, file_names, n_images=5, class_names=unified_emotion_mapping)
#     break  # Only show the first batch


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = torch.device('cpu')
print(device)

In [ ]:
resnet18p = models.resnet18(pretrained=True)
num_ftrs = resnet18p.fc.in_features
resnet18p.fc = torch.nn.Linear(num_ftrs, 7)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, downsample):
        super().__init__()
        if downsample:
            self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1)
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=2),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
            self.shortcut = nn.Sequential()

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)

    def forward(self, input):
        shortcut = self.shortcut(input)
        input = nn.ReLU()(self.bn1(self.conv1(input)))
        input = nn.ReLU()(self.bn2(self.conv2(input)))
        input = input + shortcut
        return nn.ReLU()(input)

In [ ]:
class ResNet18(nn.Module):
    def __init__(self, in_channels, resblock, outputs=7):
        super().__init__()
        self.layer0 = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.layer1 = nn.Sequential(
            resblock(64, 64, downsample=False),
            resblock(64, 64, downsample=False)
        )

        self.layer2 = nn.Sequential(
            resblock(64, 128, downsample=True),
            resblock(128, 128, downsample=False)
        )

        self.layer3 = nn.Sequential(
            resblock(128, 256, downsample=True),
            resblock(256, 256, downsample=False)
        )


        self.layer4 = nn.Sequential(
            resblock(256, 512, downsample=True),
            resblock(512, 512, downsample=False)
        )

        self.gap = torch.nn.AdaptiveAvgPool2d(1)
        self.fc = torch.nn.Linear(512, outputs)

    def forward(self, input):
        input = self.layer0(input)
        input = self.layer1(input)
        input = self.layer2(input)
        input = self.layer3(input)
        input = self.layer4(input)
        input = self.gap(input)
        input = torch.flatten(input, start_dim=1)
        input = self.fc(input)

        return input

In [ ]:
resnet18 = ResNet18(3, ResBlock, outputs=7)
resnet18.to(torch.device("cuda:0" if torch.cuda.is_available() else "cpu"))

In [ ]:
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(resnet50.parameters(), lr=0.001, momentum=0.9)

lr = 1e-3
momentum = 0.9
weight_decay = 1e-4

loss = nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(resnet50.parameters(), lr=lr, weight_decay = weight_decay)
optimizer = torch.optim.Adam(resnet18.parameters(), lr=lr, weight_decay = weight_decay)

In [ ]:
# resnet50 = resnet50.to(device)
print(next(resnet18.parameters()).device)
print(torch.cuda.is_available())  # Should return True if CUDA is available
print(torch.cuda.current_device())  # Returns the index of the current device
print(torch.cuda.get_device_name(torch.cuda.current_device()))  # Returns the name of the current device
# for inputs, labels in dataloader:
#     inputs, labels = inputs.to(device), labels.to(device)
#     print(inputs.device, labels.device)
#     break  # Just to check the first batch



In [ ]:
# resnet50 = resnet50.to(device)
# resnet50.train() 
# num_epochs = 3

# print("Using Device:",device)
# for epoch in range(num_epochs):
#     running_loss = 0.0
#     correct_predictions = 0
#     total_predictions = 0
    
#     for inputs, labels in dataloader:
#         inputs = inputs.to(device)
#         labels = labels.to(device)
        
#         optimizer.zero_grad()

#         outputs = resnet50(inputs)  # 前向传播
#         loss = criterion(outputs, labels)
#         loss.backward() 
#         optimizer.step() 

#         running_loss += loss.item()

#         _, predicted = outputs.max(1) 
#         total_predictions += labels.size(0)
#         correct_predictions += (predicted == labels).sum().item() 

#     epoch_loss = running_loss / len(dataloader)
#     epoch_acc = correct_predictions / total_predictions * 100 

#     print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

In [ ]:
import my_utils as mu
def trainf(net, train_iter, test_iter, loss, num_epochs, optimizer, device):
    """Train and evaluate a model with CPU or GPU."""
    net.to(device)
    animator = mu.d2l.Animator(xlabel='epoch', xlim=[0, num_epochs],
                            legend=['train loss', 'train acc', 'test acc'])
    timer = mu.d2l.Timer()
    for epoch in range(num_epochs):
        metric = mu.d2l.Accumulator(3)  # train_loss, train_acc, num_examples
        for i, (X, y) in enumerate(train_iter):
            timer.start()
            net.train()
            optimizer.zero_grad()
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            l = loss(y_hat, y)
            l.backward()
            optimizer.step()


            with torch.no_grad():
                metric.add(l*X.shape[0], mu.d2l.accuracy(y_hat, y), X.shape[0])
            timer.stop()
            train_loss, train_acc = metric[0]/metric[2], metric[1]/metric[2]
            if (i+1) % 5 == 0:
                animator.add(epoch + i/len(train_iter),
                              (train_loss, train_acc, None))
        test_acc = mu.evaluate_accuracy_gpu(net, test_iter)
        animator.add(epoch+1, (None, None, test_acc))
        
        print(f'loss {train_loss:.3f}, train acc {train_acc:.3f}, ')
        print(f'{metric[2] * num_epochs / timer.sum():.1f} examples/sec '
            f'on {str(device)}')
        
    print(f'loss {train_loss:.3f}, train acc {train_acc:.3f}, ')
    print(f'{metric[2] * num_epochs / timer.sum():.1f} examples/sec '
        f'on {str(device)}')

In [ ]:
print('Using device:', device)
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0)) # print the type of the chosen gpu
trainf(resnet18, dataloader, valid_loader, loss, 10, optimizer, device)

In [ ]:
# loss = nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(resnet50.parameters(), lr=lr)


# for epoch in range(1):
#     for batch_idx, (data, target) in enumerate(dataloader):
#         data = data.to(device)  # Move data to GPU
#         target = target.to(device)
#         optimizer.zero_grad()
#         output = resnet50(data)
#         l = loss(output, target)
#         l.backward()
#         optimizer.step()


#     correct = 0
#     total = 0
#     with torch.no_grad():
#         for batch_idx, (data, target) in enumerate(valid_loader):
#             output = resnet50(data)
#             _, predicted = torch.max(output.data, 1)
#             total += target.size(0)
#             correct += (predicted == target).sum().item()

#     print('Epoch: {}, Loss: {:.4f}, Acc: {:.4f}'.format(
#         epoch, loss.item(), correct / total))


# import matplotlib.pyplot as plt
# loss_list = []
# acc_list = []

# plt.plot(loss_list, label='Loss')
# plt.legend()
# plt.show()

# plt.plot(acc_list, label='Acc')
# plt.legend()
# plt.show()


In [ ]:
resnet50.eval()

state = {
    'state_dict': resnet50.state_dict(),
    'optimizer': optimizer.state_dict(),
}
torch.save(state, "my_model.pth")

img = Image.open("F:\\Aff-Wild2\\Openpose\\batch1-output\\5-60-1920x1080-1\\5-60-1920x1080-1-main_000000006141_rendered.png")
img = transform(img)
img = img.unsqueeze(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img = img.to(device)
resnet50 = resnet50.to(device)

with torch.no_grad(): 
    outputs = resnet50(img)
    _, predicted = torch.max(outputs, 1)

class_names = ["Neutral","Anger","Disgust","Fear","Happy","Sad","Surprise"]
print('Predicted:', class_names[predicted.item()])